# PoliMillionaire baseline

Before running this notebook, put the files in Google Drive like this:

```
MyDrive/
└── Colab Notebooks/
    └── NLP_assignment/
        ├── poli_millionaire_clean_baseline_v2.ipynb
        └── millionaire_client/
            ├── __init__.py
            ├── client.py
            ├── auth.py
            ├── base.py
            ├── game.py
            ├── models.py
            ├── competitions.py
            ├── leaderboard.py
            └── exceptions.py
```

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

import os
import sys
import time
import re
import torch

In [ ]:
BASE_DIR = '/content/gdrive/MyDrive/NLP_assignment'
PACKAGE_DIR = os.path.join(BASE_DIR, 'millionaire_client')

print('BASE_DIR exists:', os.path.exists(BASE_DIR))
if os.path.exists(BASE_DIR):
    print('BASE_DIR contents:', os.listdir(BASE_DIR))

print('PACKAGE_DIR exists:', os.path.exists(PACKAGE_DIR))
if os.path.exists(PACKAGE_DIR):
    print('PACKAGE_DIR contents:', os.listdir(PACKAGE_DIR))

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError('BASE_DIR not found. Create the NLP_assignment folder in Drive and upload the notebook there.')

if not os.path.exists(PACKAGE_DIR):
    raise FileNotFoundError('millionaire_client folder not found inside BASE_DIR.')

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

print('Path added successfully.')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from millionaire_client import MillionaireClient
from millionaire_client.exceptions import TimeoutError, RateLimitError

In [ ]:
API_URL = (__import__("os").getenv("POLI_MILLIONAIRE_API_URL") or input("PoliMillionaire API URL: ").strip())
USERNAME = (__import__("os").getenv("POLI_MILLIONAIRE_USERNAME") or input("PoliMillionaire username: ").strip())
PASSWORD = (__import__("os").getenv("POLI_MILLIONAIRE_PASSWORD") or __import__("getpass").getpass("PoliMillionaire password: ").strip())
client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print('Logged in as:', user.username)

In [ ]:
competitions = client.competitions.list_all()
for c in competitions:
    print(c.id, c.name, c.max_levels)

COMPETITION_ID = competitions[2].id

In [ ]:
model_id = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
model.eval()

In [ ]:
def extract_letter(text):
    text = text.strip().upper()
    match = re.search(r'\b([ABCD])\b', text)
    if match:
        return match.group(1)
    if text and text[0] in 'ABCD':
        return text[0]
    return 'A'

def choose_answer(question):
    if len(question.options) < 4:
        return question.options[0].id, 'A', 'fallback'

    prompt = f'''PoliMillionaire MCQ. Reply only with A, B, C, or D.


Question: {question.text}
A) {question.options[0].text}
B) {question.options[1].text}
C) {question.options[2].text}
D) {question.options[3].text}

Answer:'''

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    letter = extract_letter(text)
    idx = ['A', 'B', 'C', 'D'].index(letter)
    return question.options[idx].id, letter, text

In [ ]:
game = client.game.start(competition_id=COMPETITION_ID, mode='text')

while game.in_progress:
    question = game.current_question
    if question is None:
        break

    print('Level:', game.current_level)
    print(question.text)
    for i, opt in enumerate(question.options):
        print(f"{chr(65+i)}) {opt.text}")

    option_id, letter, raw = choose_answer(question)
    print('Predicted:', letter, '| Raw output:', raw)

    try:
        result = game.answer(option_id)
    except TimeoutError:
        print('Timed out')
        break
    except RateLimitError:
        print('Rate limited, waiting...')
        time.sleep(5)
        result = game.answer(option_id)

    print('Correct:', result.correct, '| Earned:', result.earned_amount)

    if result.game_over:
        break

    time.sleep(0.5)

print('Final earned:', game.earned_amount)